In [1]:
# Load libraries
library("anndata")
library("DESeq2")
library("reticulate")

# Use the path to your Python executable in the virtual environment
use_python("/lila/home/forsythb/.virtualenvs/r-reticulate/bin/")

# Look at where Python is located
py_config()

# Import scanpy
sc <- import("scanpy")

Loading required package: S4Vectors

Warning message:
“package ‘S4Vectors’ was built under R version 4.3.2”
Loading required package: stats4

Loading required package: BiocGenerics

Warning message:
“package ‘BiocGenerics’ was built under R version 4.3.2”

Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min



Attaching package: ‘S4Vectors’


The following object is masked from ‘package:utils’:

    findMatches


The following objects are masked from ‘package:base’:

   

python:         /lila/home/forsythb/.virtualenvs/r-reticulate/bin/python
libpython:      /home/forsythb/anaconda3/lib/libpython3.8.so
pythonhome:     /lila/home/forsythb/.virtualenvs/r-reticulate:/lila/home/forsythb/.virtualenvs/r-reticulate
version:        3.8.18 (default, Sep 11 2023, 13:40:15)  [GCC 11.2.0]
numpy:          /lila/home/forsythb/.virtualenvs/r-reticulate/lib/python3.8/site-packages/numpy
numpy_version:  1.24.4

NOTE: Python version was forced by use_python() function

In [4]:
# Read in the adata
adata <- sc$read_h5ad("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/metacells/metacells.020524/adata.post.combined.h5ad")

In [5]:
adata$obs

,Patient,Tumor_Site,Culture_Media,ZFP_Expression,Sample_Name,Batch,phenograph
,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>,<fct>
146_P_HISC_CTRL_1_SEACell-125,146,Primary,HISC,CTRL,146_P_HISC_CTRL_1,6,15
146_P_HISC_CTRL_1_SEACell-9,146,Primary,HISC,CTRL,146_P_HISC_CTRL_1,6,0
146_P_BASE_CTRL_1_SEACell-52,146,Primary,BASE,CTRL,146_P_BASE_CTRL_1,5,1
146_M_BASE_CTRL_1_SEACell-109,146,Metastatic,BASE,CTRL,146_M_BASE_CTRL_1,2,2
146_P_Dediff_ZFPKD_1_SEACell-35,146,Primary,Dedifferentiated,ZFPKD,146_P_Dediff_ZFPKD_1,8,4
146_P_Dediff_ZFPKD_2_SEACell-34,146,Primary,Dedifferentiated,ZFPKD,146_P_Dediff_ZFPKD_2,8,8
146_P_Dediff_ZFPKD_2_SEACell-85,146,Primary,Dedifferentiated,ZFPKD,146_P_Dediff_ZFPKD_2,8,8
146_M_BASE_CTRL_1_SEACell-115,146,Metastatic,BASE,CTRL,146_M_BASE_CTRL_1,2,10
146_P_HISC_CTRL_1_SEACell-176,146,Primary,HISC,CTRL,146_P_HISC_CTRL_1,6,0


In [8]:
# Extract count matrix from the raw layer
count_matrix <- adata$raw$X
count_matrix <- t(count_matrix)
count_matrix <- round(count_matrix)

In [9]:
count_matrix

0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
2,2,3,4,4,4,4,5,3,4,⋯,0,3,3,4,4,3,4,4,3,3
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,1,0,0,1
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,1,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,1,0,0,0
0,0,0,0,0,0,0,0,0,0,⋯,0,0,0,0,0,0,0,0,0,0


In [10]:
# Define the column data
coldata <- adata$obs[,c("Tumor_Site","Culture_Media", "ZFP_Expression", "Batch")]

In [11]:
# Convert column data to factors
coldata$Tumor_Site <- factor(coldata$Tumor_Site)
coldata$Culture_Media <- factor(coldata$Culture_Media)
coldata$ZFP_Expression <- factor(coldata$ZFP_Expression)
coldata$Batch <- factor(coldata$Batch)

In [12]:
# Design a DESeqDataSet
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + ZFP_Expression
)

converting counts to integer mode



In [13]:
# Add a pseudo-count of 1 to all counts
count_matrix <- counts(dds_1) + 1

# Specify the row names as the gene names
rownames(count_matrix) <- adata$raw$var_names

# Set the reference levels
coldata$Tumor_Site<-relevel(coldata$Tumor_Site,ref="Primary")
coldata$Culture_Media<-relevel(coldata$Culture_Media,ref="BASE")
coldata$ZFP_Expression<-relevel(coldata$ZFP_Expression,ref="CTRL")

# Design a DESeqDataSet with modified count matrix
dds_1 <- DESeqDataSetFromMatrix(
  countData = count_matrix,
  colData = coldata,
  design = ~ Tumor_Site + Culture_Media + ZFP_Expression + ZFP_Expression:Culture_Media  
)

# Perform DESeq analysis
# dds_1 <- DESeq(dds_1)

# # Extract results
# result_names_1 <- resultsNames(dds_1)
# result_names_1

# # Specify the contrast and extract results
# res_1 <- results(dds_1)
# res_1

converting counts to integer mode



In [35]:
# Specify the date
date <- '020224'

# Specify the directory path
directory_path <- paste("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/deseq/deseq.", date, sep="")

# Check if the directory exists, and create it if not
if (!dir.exists(directory_path)) {
  dir.create(directory_path, recursive = TRUE)
}

# Specify the file path where you want to save the results
result_file <- paste(directory_path, "/metacells_zfpexp_intcm_results.csv", sep="")

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = paste(directory_path, "/metacells_zfpexp_intcm_dds.rds", sep=""))

# Save the DESeq results
saveRDS(results(dds_1), file = paste(directory_path, "/metacells_zfpexp_intcm_results.rds", sep=""))


In [ ]:
res1 <- results(dds, contrast = c("Type", "1020A", "lactate"))

ddsLRT <- DESeq(dds, test="LRT", reduced= ~ CellType)
resLRT <- results(ddsLRT)

In [39]:
full_model <- ~ Tumor_Site + Culture_Media + ZFP_Expression + ZFP_Expression:Culture_Media
reduced_model <- ~ Tumor_Site + Culture_Media + ZFP_Expression

In [40]:
dds <- DESeqDataSetFromMatrix(countData = count_matrix, colData = coldata, design = ~ Tumor_Site + Culture_Media + ZFP_Expression + ZFP_Expression:Culture_Media)
dds_lrt <- DESeq(dds, test="LRT", reduced = ~ Tumor_Site + Culture_Media + ZFP_Expression)

converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

-- note: fitType='parametric', but the dispersion trend was not well captured by the
   function: y = a/x + b, and a local regression fit was automatically substituted.
   specify fitType='local' or 'mean' to avoid this message next time.

final dispersion estimates

fitting model and testing

-- replacing outliers and refitting for 18 genes
-- DESeq argument 'minReplicatesForReplace' = 7 
-- original counts are preserved in counts(dds)

estimating dispersions

fitting model and testing



In [41]:
# Specify the date
date <- '020224'

# Specify the directory path
directory_path <- paste("/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/deseq/deseq.", date, sep="")

# Check if the directory exists, and create it if not
if (!dir.exists(directory_path)) {
  dir.create(directory_path, recursive = TRUE)
}

# Specify the file path where you want to save the results
result_file <- paste(directory_path, "/metacells_zfpexp_intcm_lrt_results.csv", sep="")

# Write the results table to a CSV file
write.csv(as.data.frame(res_1), file = result_file)

# Save the DESeqDataSet object
saveRDS(dds_1, file = paste(directory_path, "/metacells_zfpexp_intcm_lrt_dds.rds", sep=""))

# Save the DESeq results
saveRDS(results(dds_1), file = paste(directory_path, "/metacells_zfpexp_intcm_lrt_results.rds", sep=""))